In [6]:
import pandas as pd
import os

RAW_DIR = "./data/raw"

# List every file so we work from the actual folder, not memory
files = [f for f in os.listdir(RAW_DIR) if f.endswith(('.csv', '.xlsx'))]
print(f"Found {len(files)} files:")
for f in files:
    print(" -", f)

Found 15 files:
 - CB-Insights_Global-Unicorn-Club.xlsx
 - funding_rounds.csv
 - investors.csv
 - startup_data.csv
 - unicorn_companies.csv
 - Indian_Startup_Funding_Dataset.csv
 - Startup_funding_2025.csv
 - quarterly_summary.csv
 - yc_startups.csv
 - startup_success_dataset.csv
 - openvc_investors.csv
 - Recently Funded Startups In India 2026.csv
 - startups.csv
 - startup_funding_dataset (1).csv
 - ai_startup_funding.csv


In [7]:
# ===== SETUP CELL — run this first, every session =====
import pandas as pd
import numpy as np
import os
import re
from io import StringIO

RAW_DIR = "../data/raw"


def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)

    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df


def parse_amount_to_usd(val, inr_to_usd=0.012):
    if pd.isna(val) or val == '-' or 'undisclosed' in str(val).lower():
        return None
    val = str(val).strip()
    if 'crore' in val.lower():
        num = float(re.search(r'[\d.]+', val).group())
        return num * 1e7 * inr_to_usd
    elif '$' in val:
        num = float(re.search(r'[\d.]+', val.replace(',', '')).group())
        if 'K' in val.upper():
            multiplier = 1e3
        elif 'M' in val.upper():
            multiplier = 1e6
        elif 'B' in val.upper():
            multiplier = 1e9
        else:
            multiplier = 1
        return num * multiplier
    return None


def clean_dollar_string(val):
    if pd.isna(val):
        return None
    return float(str(val).replace('$', '').replace(',', ''))


def standardize(df, name_col, industry_col, country_col, amount_col, stage_col, source_name):
    return pd.DataFrame({
        'startup_name': df[name_col],
        'industry': df[industry_col],
        'country': df[country_col],
        'funding_amount_usd': df[amount_col],
        'funding_stage': df[stage_col] if stage_col else None,
        'source_dataset': source_name
    })
# ========================================================

In [29]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("yc_startups.csv")


yc_startups.csv
Shape: (999, 9)

Columns:
['name', 'batch', 'location', 'description', 'tags', 'yc_profile_url', 'website', 'status', 'team_size']

Dtypes:
name               object
batch              object
location           object
description        object
tags               object
yc_profile_url     object
website           float64
status            float64
team_size         float64
dtype: object

Null counts:
batch            1
location       191
description      6
website        999
status         999
team_size      999
dtype: int64

Sample rows:


,name,batch,location,description,tags,yc_profile_url,website,status,team_size
0,DoorDash,Summer 2013,"San Francisco, CA, USA",Restaurant delivery.,"CONSUMER, FOOD AND BEVERAGE",https://www.ycombinator.com/companies/doordash,NaN,NaN,NaN
1,Airbnb,Winter 2009,"San Francisco, CA, USA",Book accommodations around the world.,"CONSUMER, TRAVEL, LEISURE AND TOURISM",https://www.ycombinator.com/companies/airbnb,NaN,NaN,NaN
2,Coinbase,Summer 2012,"Los Angeles, CA, USA","Buy, sell, and manage cryptocurrencies.","FINTECH, BANKING AND EXCHANGE",https://www.ycombinator.com/companies/coinbase,NaN,NaN,NaN


In [30]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("CB-Insights_Global-Unicorn-Club.xlsx")


CB-Insights_Global-Unicorn-Club.xlsx
Shape: (1385, 8)

Columns:
['Unnamed: 0', 'Global Unicorn Club: Private Companies Valued at $1B+\n(as of April 2026)', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']

Dtypes:
Unnamed: 0                                                                   float64
Global Unicorn Club: Private Companies Valued at $1B+\n(as of April 2026)     object
Unnamed: 2                                                                    object
Unnamed: 3                                                                    object
Unnamed: 4                                                                    object
Unnamed: 5                                                                    object
Unnamed: 6                                                                    object
Unnamed: 7                                                                    object
dtype: object

Null counts:
Unnamed: 0                                

,Unnamed: 0,Global Unicorn Club: Private Companies Valued at $1B+\n(as of April 2026),Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors
2,NaN,OpenAI,840,2019-07-22 00:00:00,United States,San Francisco,Enterprise Tech,"Khosla Ventures, Thrive Capital, Sequoia Capital"


In [31]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("funding_rounds.csv")


funding_rounds.csv
Shape: (2818, 18)

Columns:
['round_id', 'startup_id', 'startup_name', 'funding_date', 'year', 'quarter', 'funding_stage', 'amount_million_usd', 'pre_money_valuation_million_usd', 'post_money_valuation_million_usd', 'equity_dilution_pct', 'lead_investor', 'investor_type', 'num_investors', 'sector', 'country', 'region', 'city']

Dtypes:
round_id                             object
startup_id                           object
startup_name                         object
funding_date                         object
year                                  int64
quarter                              object
funding_stage                        object
amount_million_usd                  float64
pre_money_valuation_million_usd     float64
post_money_valuation_million_usd    float64
equity_dilution_pct                 float64
lead_investor                        object
investor_type                        object
num_investors                         int64
sector                    

,round_id,startup_id,startup_name,funding_date,year,quarter,funding_stage,amount_million_usd,pre_money_valuation_million_usd,post_money_valuation_million_usd,equity_dilution_pct,lead_investor,investor_type,num_investors,sector,country,region,city
0,R00001,S00001,DataTech,2023-03-22,2023,Q1,Seed,4.20,13.98,18.18,13.8,Redpoint Ventures,Angel Investor,1,Gaming & Entertainment,Mexico,South America,Mexico City
1,R00002,S00002,VertexPlatform,2025-01-19,2025,Q1,Pre-Seed,0.55,2.34,2.89,12.7,General Catalyst,Angel Investor,2,Artificial Intelligence,Indonesia,Asia,Surabaya
2,R00003,S00002,VertexPlatform,2021-07-14,2021,Q3,Series A,15.35,176.14,191.49,18.4,Index Ventures,Corporate VC,3,Artificial Intelligence,Indonesia,Asia,Surabaya


In [32]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("investors.csv")


investors.csv
Shape: (73, 10)

Columns:
['investor_name', 'investor_type', 'hq_country', 'founded_year', 'aum_billion_usd', 'total_deals_in_dataset', 'total_invested_million_usd', 'avg_deal_size_million_usd', 'top_sector', 'top_stage']

Dtypes:
investor_name                  object
investor_type                  object
hq_country                     object
founded_year                    int64
aum_billion_usd               float64
total_deals_in_dataset          int64
total_invested_million_usd    float64
avg_deal_size_million_usd     float64
top_sector                     object
top_stage                      object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,investor_name,investor_type,hq_country,founded_year,aum_billion_usd,total_deals_in_dataset,total_invested_million_usd,avg_deal_size_million_usd,top_sector,top_stage
0,Sequoia Capital,Angel Investor,United States,1997,25.2,29,999.88,34.48,E-Commerce,Pre-Seed
1,Andreessen Horowitz (a16z),Angel Investor,United States,1979,2.5,43,526.07,12.23,Artificial Intelligence,Pre-Seed
2,Tiger Global,Angel Investor,United States,1963,22.4,37,120.42,3.25,Artificial Intelligence,Seed


In [33]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("startup_data.csv")


startup_data.csv
Shape: (500, 12)

Columns:
['Startup Name', 'Industry', 'Funding Rounds', 'Funding Amount (M USD)', 'Valuation (M USD)', 'Revenue (M USD)', 'Employees', 'Market Share (%)', 'Profitable', 'Year Founded', 'Region', 'Exit Status']

Dtypes:
Startup Name               object
Industry                   object
Funding Rounds              int64
Funding Amount (M USD)    float64
Valuation (M USD)         float64
Revenue (M USD)           float64
Employees                   int64
Market Share (%)          float64
Profitable                  int64
Year Founded                int64
Region                     object
Exit Status                object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,Startup Name,Industry,Funding Rounds,Funding Amount (M USD),Valuation (M USD),Revenue (M USD),Employees,Market Share (%),Profitable,Year Founded,Region,Exit Status
0,Startup_1,IoT,1,101.09,844.75,67.87,1468,5.20,0,2006,Europe,Private
1,Startup_2,EdTech,1,247.62,3310.83,75.65,3280,8.10,1,2003,South America,Private
2,Startup_3,EdTech,1,109.24,1059.37,84.21,4933,2.61,1,1995,South America,Private


In [34]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("unicorn_companies.csv")


unicorn_companies.csv
Shape: (1500, 11)

Columns:
['Company', 'Valuation ($B)', 'Date Joined', 'Country', 'City', 'Industry', 'Select Investors', 'Founded Year', 'Total Raised ($B)', 'Financial Stage', 'Investors Count']

Dtypes:
Company               object
Valuation ($B)       float64
Date Joined           object
Country               object
City                  object
Industry              object
Select Investors      object
Founded Year           int64
Total Raised ($B)    float64
Financial Stage       object
Investors Count        int64
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors,Founded Year,Total Raised ($B),Financial Stage,Investors Count
0,Grid Group94,1.58,2026-12-04,United States,Los Angeles,Data management & analytics,"DST Global, Kleiner Perkins Caufield & Byers, ...",2016,0.305,Series E,5
1,Meta Labs,2.55,2026-11-26,Germany,Munich,Fintech,Y Combinator,2022,0.528,Series D,12
2,Link Group,2.32,2026-10-14,Israel,Tel Aviv,Health,"New Enterprise Associates, DST Global, Alibaba...",2015,0.940,Series C,16


In [35]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("Indian_Startup_Funding_Dataset.csv")


Indian_Startup_Funding_Dataset.csv
Shape: (300, 11)

Columns:
['Startup_ID', 'Startup_Name', 'Industry', 'City', 'State', 'Funding_Year', 'Funding_Stage', 'Lead_Investor', 'Funding_Amount_USD', 'Employees', 'Status']

Dtypes:
Startup_ID             int64
Startup_Name          object
Industry              object
City                  object
State                 object
Funding_Year           int64
Funding_Stage         object
Lead_Investor         object
Funding_Amount_USD     int64
Employees              int64
Status                object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,Startup_ID,Startup_Name,Industry,City,State,Funding_Year,Funding_Stage,Lead_Investor,Funding_Amount_USD,Employees,Status
0,1,Startup_001,Cybersecurity,Pune,Maharashtra,2025,Series A,Blume Ventures,20300000,417,Active
1,2,Startup_002,E-commerce,Pune,Maharashtra,2025,Pre-Series A,Accel,12200000,406,Active
2,3,Startup_003,AgriTech,Ahmedabad,Gujarat,2022,Seed,Accel,8300000,362,Active


In [36]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("Startup_funding_2025.csv")


Startup_funding_2025.csv
Shape: (887, 6)

Columns:
['Company', 'Sector', 'Headquarters', 'Amount', 'Funding_Round_Type', 'Lead_Investors']

Dtypes:
Company               object
Sector                object
Headquarters          object
Amount                object
Funding_Round_Type    object
Lead_Investors        object
dtype: object

Null counts:
Sector    1
dtype: int64

Sample rows:


,Company,Sector,Headquarters,Amount,Funding_Round_Type,Lead_Investors
0,Unicommerce,SaaS,New Delhi,Rs 124.5 crore,-,Anchor investors
1,Banana Club,Consumer > Fashion Tech,Karnataka,$1.39M,Seed,-
2,Neulife,HealthTech > Fitness & Wellness Tech,Maharashtra,$1M,Unattributed,Subhkam Ventures


In [37]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("quarterly_summary.csv")


quarterly_summary.csv
Shape: (25, 10)

Columns:
['year', 'quarter', 'total_deals', 'total_funding_million_usd', 'avg_deal_size_million_usd', 'median_deal_size_million_usd', 'median_valuation_million_usd', 'unique_startups', 'unique_investors', 'period']

Dtypes:
year                              int64
quarter                          object
total_deals                       int64
total_funding_million_usd       float64
avg_deal_size_million_usd       float64
median_deal_size_million_usd    float64
median_valuation_million_usd    float64
unique_startups                   int64
unique_investors                  int64
period                           object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,year,quarter,total_deals,total_funding_million_usd,avg_deal_size_million_usd,median_deal_size_million_usd,median_valuation_million_usd,unique_startups,unique_investors,period
0,2020,Q1,134,1488.06,11.10,2.24,16.64,131,61,2020-Q1
1,2020,Q2,115,901.77,7.84,2.07,14.39,112,57,2020-Q2
2,2020,Q3,127,2414.86,19.01,1.87,13.31,124,60,2020-Q3


In [38]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("startup_success_dataset.csv")


startup_success_dataset.csv
Shape: (100000, 11)

Columns:
['funding_rounds', 'founder_experience_years', 'team_size', 'market_size_billion', 'product_traction_users', 'burn_rate_million', 'revenue_million', 'investor_type', 'sector', 'founder_background', 'outcome']

Dtypes:
funding_rounds                int64
founder_experience_years      int64
team_size                     int64
market_size_billion         float64
product_traction_users        int64
burn_rate_million           float64
revenue_million             float64
investor_type                object
sector                       object
founder_background           object
outcome                      object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,funding_rounds,founder_experience_years,team_size,market_size_billion,product_traction_users,burn_rate_million,revenue_million,investor_type,sector,founder_background,outcome
0,4,13,58,48.225483,594843,18.519211,1.483962e+06,tier2_vc,Health,academic,IPO
1,1,6,221,31.532647,393020,14.298149,8.620568e+05,tier2_vc,Fintech,first_time,Failure
2,3,5,247,4.969722,27636,20.447567,9.726169e+04,none,SaaS,first_time,Failure


In [39]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("openvc_investors.csv")


openvc_investors.csv
Shape: (200, 12)

Columns:
['name', 'firm_type', 'countries', 'check_size', 'stages', 'lead_status', 'thesis', 'value_add', 'typical_terms', 'reply_rate', 'profile_url', 'source_category']

Dtypes:
name               object
firm_type          object
countries          object
check_size         object
stages             object
lead_status        object
thesis             object
value_add          object
typical_terms      object
reply_rate         object
profile_url        object
source_category    object
dtype: object

Null counts:
lead_status        3
value_add        117
typical_terms    199
reply_rate       180
dtype: int64

Sample rows:


,name,firm_type,countries,check_size,stages,lead_status,thesis,value_add,typical_terms,reply_rate,profile_url,source_category
0,Tunitas Ventures,VC firm,USA,$500k to $1M,3. Early Revenue,Sometimes,We invest in category creating companies acros...,Our main value-add is helping our portfolio co...,NaN,50%,https://www.openvc.app/fund/Tunitas%20Ventures,venture-capital-firms
1,Maniv Mobility,VC firm,"Brazil, Canada",$1M to $5M,"1. Idea or Patent, 2. Prototype",NaN,We invest in the world's leading mobility star...,Extensive network within the broader mobility ...,NaN,50%,https://www.openvc.app/fund/Maniv%20Mobility,venture-capital-firms
2,Sentiero Ventures,VC firm,"Canada, USA",$200k to $500k,"2. Prototype, 3. Early Revenue",Sometimes,We invest in B2B AI-enabled SaaS startups that...,"With a venture capital investment from us, sta...",NaN,86%,https://www.openvc.app/fund/Sentiero%20Ventures,venture-capital-firms


In [40]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("Recently Funded Startups In India 2026.csv")


Recently Funded Startups In India 2026.csv
Shape: (100, 7)

Columns:
['Name', 'Website', 'Industry', 'Country', 'Funding Amount (USD)', 'Funding Type', 'Last Funding Date']

Dtypes:
Name                    object
Website                 object
Industry                object
Country                 object
Funding Amount (USD)    object
Funding Type            object
Last Funding Date       object
dtype: object

Null counts:
Funding Amount (USD)    13
dtype: int64

Sample rows:


,Name,Website,Industry,Country,Funding Amount (USD),Funding Type,Last Funding Date
0,EarthSync,earthsync.io,"Energy, Analytics, Artificial Intelligence, B2...",India,"$1,000,000",Pre-Seed,February 2026
1,4baseCare,4basecare.com,"Healthcare, Artificial Intelligence, B2B Softw...",India,"$9,803,025",Series B,February 2026
2,Cava Athleisure,cavaathleisure.com,"Fashion, Wellness, Consumer Goods, E-commerce,...",India,"$4,353,085",Series A,February 2026


In [41]:
def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)
    
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df

# Run this for EVERY file, one at a time — don't skip any
df_yc = inspect("startups.csv")


startups.csv
Shape: (1747, 10)

Columns:
['startup_id', 'startup_name', 'sector', 'country', 'region', 'city', 'founded_year', 'employee_count', 'female_founder', 'revenue_status']

Dtypes:
startup_id        object
startup_name      object
sector            object
country           object
region            object
city              object
founded_year       int64
employee_count     int64
female_founder     int64
revenue_status    object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,startup_id,startup_name,sector,country,region,city,founded_year,employee_count,female_founder,revenue_status
0,S00001,DataTech,Gaming & Entertainment,Mexico,South America,Mexico City,2020,9,0,Profitable
1,S00002,VertexPlatform,Artificial Intelligence,Indonesia,Asia,Surabaya,2024,2,1,Pre-Revenue
2,S00003,KiteAI,SaaS / Enterprise Software,Germany,Europe,Munich,2019,4,0,Pre-Revenue


In [42]:
def inspect(filename, nrows=None):

    path = os.path.join(RAW_DIR, filename)

    if filename.endswith('.xlsx'):

        df = pd.read_excel(path, nrows=nrows)

    else:

        df = pd.read_csv(path, nrows=nrows, low_memory=False)

    

    print(f"\n{'='*60}\n{filename}\n{'='*60}")

    print(f"Shape: {df.shape}")

    print(f"\nColumns:\n{list(df.columns)}")

    print(f"\nDtypes:\n{df.dtypes}")

    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

    print(f"\nSample rows:")

    display(df.head(3))

    return df

# Run this for EVERY file, one at a time — don't skip any

df_yc = inspect("startup_funding_dataset (1).csv")


startup_funding_dataset (1).csv
Shape: (2000, 7)

Columns:
['Startup Name', 'Industry', 'Country', 'Funding Stage', 'Amount Raised (USD)', 'Funding Date', 'Number of Employees']

Dtypes:
Startup Name            object
Industry                object
Country                 object
Funding Stage           object
Amount Raised (USD)    float64
Funding Date            object
Number of Employees      int64
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,Startup Name,Industry,Country,Funding Stage,Amount Raised (USD),Funding Date,Number of Employees
0,"Williamson, Greer and Clark",SaaS,Australia,Seed,304706.0,2016-07-03,386
1,"Bradford, Green and Miranda",Finance,India,IPO,641212096.0,2025-08-10,474
2,Kelly LLC,AI,UK,Series A,4149256.0,2024-11-13,220


In [43]:
def inspect(filename, nrows=None):

    path = os.path.join(RAW_DIR, filename)

    if filename.endswith('.xlsx'):

        df = pd.read_excel(path, nrows=nrows)

    else:

        df = pd.read_csv(path, nrows=nrows, low_memory=False)

    

    print(f"\n{'='*60}\n{filename}\n{'='*60}")

    print(f"Shape: {df.shape}")

    print(f"\nColumns:\n{list(df.columns)}")

    print(f"\nDtypes:\n{df.dtypes}")

    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

    print(f"\nSample rows:")

    display(df.head(3))

    return df

# Run this for EVERY file, one at a time — don't skip any

df_yc = inspect("ai_startup_funding.csv")


ai_startup_funding.csv
Shape: (60, 1)

Columns:
['Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type']

Dtypes:
Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type    object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,"Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type"
0,"DeepMind Health,Healthcare Diagnostics,UK,Seri..."
1,"ScaleAI,Autonomous Vehicles,USA,Series E,7300,..."
2,"Synthesia,Synthetic Media,UK,Series B,1200,90,..."


In [44]:
# Peek at the raw first line to see what's actually separating fields
with open(os.path.join(RAW_DIR, "ai_startup_funding.csv"), 'r', encoding='utf-8') as f:
    print(f.readline())
    print(f.readline())

"Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type"

"DeepMind Health,Healthcare Diagnostics,UK,Series D,4500,300,""Google, Sequoia"",2020,FALSE,1200,Reinforcement Learning"



In [46]:

df_ai = pd.read_csv(os.path.join(RAW_DIR, "ai_startup_funding.csv"), sep=';')

In [47]:
# Peek at the raw first line to see what's actually separating fields
with open(os.path.join(RAW_DIR, "ai_startup_funding.csv"), 'r', encoding='utf-8') as f:
    print(f.readline())
    print(f.readline())

"Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type"

"DeepMind Health,Healthcare Diagnostics,UK,Series D,4500,300,""Google, Sequoia"",2020,FALSE,1200,Reinforcement Learning"



In [48]:
from io import StringIO

path = os.path.join(RAW_DIR, "ai_startup_funding.csv")

with open(path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Strip the outer wrapping quote from each line, then unescape doubled quotes
cleaned_lines = []
for line in lines:
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    line = line.replace('""', '"')
    cleaned_lines.append(line)

csv_text = "\n".join(cleaned_lines)
df_ai = pd.read_csv(StringIO(csv_text))
df_ai.head()

,Startup_Name,Industry_AI_Application,Country,Funding_Stage,Valuation ($M),Funding_Amount ($M),Investors,Year,Profitability,Employee_Count,AI_Model_Type
0,DeepMind Health,Healthcare Diagnostics,UK,Series D,4500,300,"Google, Sequoia",2020,False,1200,Reinforcement Learning
1,ScaleAI,Autonomous Vehicles,USA,Series E,7300,500,"Y Combinator, Tiger Global",2023,True,850,Computer Vision
2,Synthesia,Synthetic Media,UK,Series B,1200,90,Kleiner Perkins,2022,False,150,Generative AI
3,Hugging Face,NLP Tools,USA,Series C,4000,200,"Lux Capital, A16Z",2021,False,300,Transformer Models
4,Waymo,Self-Driving Cars,USA,Series F,30000,1000,"Alphabet, Silver Lake",2023,False,2500,Deep Learning


In [49]:
df_cb = pd.read_excel(
    os.path.join(RAW_DIR, "CB-Insights_Global-Unicorn-Club.xlsx"),
    skiprows=2  # skip the title row + blank row, header becomes 'Company', 'Valuation ($B)', etc.
)
df_cb.head()

,Unnamed: 0,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors
0,NaN,OpenAI,840.0,2019-07-22 00:00:00,United States,San Francisco,Enterprise Tech,"Khosla Ventures, Thrive Capital, Sequoia Capital"
1,NaN,ByteDance,480.0,2017-04-07 00:00:00,China,Beijing,Media & Entertainment,"Sequoia Capital China, SIG Asia Investments, S..."
2,NaN,SpaceX,400.0,2012-12-01 00:00:00,United States,Hawthorne,Industrials,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
3,NaN,Anthropic,380.0,2023-02-03 00:00:00,United States,San Francisco,Enterprise Tech,Google
4,NaN,Stripe,159.0,2014-01-23 00:00:00,United States,San Francisco,Financial Services,"Khosla Ventures, LowercaseCapital, capitalG"


In [ ]:
df_2025 = pd.read_csv(os.path.join(RAW_DIR, "Startup_funding_2025.csv"))
df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)
df_2025[['Amount', 'amount_usd']].head(10)

In [52]:
df_cb = df_cb.drop(columns=['Unnamed: 0'])

In [53]:
df_cb

,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors
0,OpenAI,840.0,2019-07-22 00:00:00,United States,San Francisco,Enterprise Tech,"Khosla Ventures, Thrive Capital, Sequoia Capital"
1,ByteDance,480.0,2017-04-07 00:00:00,China,Beijing,Media & Entertainment,"Sequoia Capital China, SIG Asia Investments, S..."
2,SpaceX,400.0,2012-12-01 00:00:00,United States,Hawthorne,Industrials,"Founders Fund, Draper Fisher Jurvetson, Rothen..."
3,Anthropic,380.0,2023-02-03 00:00:00,United States,San Francisco,Enterprise Tech,Google
4,Stripe,159.0,2014-01-23 00:00:00,United States,San Francisco,Financial Services,"Khosla Ventures, LowercaseCapital, capitalG"
...,...,...,...,...,...,...,...
1378,NewsletterResearch PortalAI 100Digital Health ...,NaN,NaN,NaN,NaN,NaN,NaN
1379,Corporate InnovationCorporate StrategyCorporat...,NaN,NaN,NaN,NaN,NaN,NaN
1380,PricingPrivacy PolicyTerms of ServicePartnersh...,NaN,NaN,NaN,NaN,NaN,NaN
1381,498 Seventh Avenue 12th Floor,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def parse_amount_to_usd(val, inr_to_usd=0.012):
    if pd.isna(val) or val == '-' or 'undisclosed' in str(val).lower():
        return None
    val = str(val).strip()
    if 'crore' in val.lower():
        num = float(re.search(r'[\d.]+', val).group())
        return num * 1e7 * inr_to_usd
    elif '$' in val:
        num = float(re.search(r'[\d.]+', val.replace(',', '')).group())
        if 'K' in val.upper():
            multiplier = 1e3
        elif 'M' in val.upper():
            multiplier = 1e6
        elif 'B' in val.upper():
            multiplier = 1e9
        else:
            multiplier = 1
        return num * multiplier
    return None

df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)
df_2025[['Amount', 'amount_usd']].head(10)

In [55]:
# A real unicorn row always has a Valuation - use that as the anchor to filter
df_cb_clean = df_cb[df_cb['Valuation ($B)'].notna()].reset_index(drop=True)

print(f"Before: {df_cb.shape[0]} rows → After: {df_cb_clean.shape[0]} rows")
df_cb_clean.tail()  # confirm the junk rows are gone

Before: 1383 rows → After: 1360 rows


,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors
1355,LeadSquared,1.0,2022-06-21 00:00:00,India,Bengaluru,Enterprise Tech,"Gaja Capital Partners, Stakeboat Capital, West..."
1356,FourKites,1.0,2022-06-21 00:00:00,United States,Chicago,Enterprise Tech,"Hyde Park Venture Partners, Bain Capital Ventu..."
1357,VulcanForms,1.0,2022-07-05 00:00:00,United States,Burlington,Industrials,"Eclipse Ventures, D1 Capital Partners, Industr..."
1358,SingleStore,1.0,2022-07-12 00:00:00,United States,San Francisco,Enterprise Tech,"Google Ventures, Accel, Data Collective"
1359,Unstoppable Domains,1.0,2022-07-27 00:00:00,United States,Las Vegas,Media & Entertainment,"Boost VC, Draper Associates, Gaingels"


In [56]:
df_investors = inspect("investors.csv")  # re-run if kernel lost it

df_investors = df_investors.reset_index(drop=True)
df_investors.insert(0, 'investor_id', ['INV' + str(i+1).zfill(4) for i in range(len(df_investors))])

df_investors.head()


investors.csv
Shape: (73, 10)

Columns:
['investor_name', 'investor_type', 'hq_country', 'founded_year', 'aum_billion_usd', 'total_deals_in_dataset', 'total_invested_million_usd', 'avg_deal_size_million_usd', 'top_sector', 'top_stage']

Dtypes:
investor_name                  object
investor_type                  object
hq_country                     object
founded_year                    int64
aum_billion_usd               float64
total_deals_in_dataset          int64
total_invested_million_usd    float64
avg_deal_size_million_usd     float64
top_sector                     object
top_stage                      object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,investor_name,investor_type,hq_country,founded_year,aum_billion_usd,total_deals_in_dataset,total_invested_million_usd,avg_deal_size_million_usd,top_sector,top_stage
0,Sequoia Capital,Angel Investor,United States,1997,25.2,29,999.88,34.48,E-Commerce,Pre-Seed
1,Andreessen Horowitz (a16z),Angel Investor,United States,1979,2.5,43,526.07,12.23,Artificial Intelligence,Pre-Seed
2,Tiger Global,Angel Investor,United States,1963,22.4,37,120.42,3.25,Artificial Intelligence,Seed


,investor_id,investor_name,investor_type,hq_country,founded_year,aum_billion_usd,total_deals_in_dataset,total_invested_million_usd,avg_deal_size_million_usd,top_sector,top_stage
0,INV0001,Sequoia Capital,Angel Investor,United States,1997,25.2,29,999.88,34.48,E-Commerce,Pre-Seed
1,INV0002,Andreessen Horowitz (a16z),Angel Investor,United States,1979,2.5,43,526.07,12.23,Artificial Intelligence,Pre-Seed
2,INV0003,Tiger Global,Angel Investor,United States,1963,22.4,37,120.42,3.25,Artificial Intelligence,Seed
3,INV0004,Accel Partners,Corporate VC,United States,1964,2.7,38,293.88,7.73,Fintech,Seed
4,INV0005,SoftBank Vision Fund,Corporate VC,India,1997,4.1,45,480.96,10.69,Artificial Intelligence,Seed


In [57]:
df_funding = inspect("funding_rounds.csv")

# Build a name -> investor_id lookup
investor_lookup = dict(zip(df_investors['investor_name'], df_investors['investor_id']))

df_funding['investor_id'] = df_funding['lead_investor'].map(investor_lookup)

# Check match rate - this is important to verify, not assume
match_rate = df_funding['investor_id'].notna().mean()
print(f"Match rate: {match_rate:.1%}")
print(f"\nUnmatched lead_investor values (top 10):")
print(df_funding[df_funding['investor_id'].isna()]['lead_investor'].value_counts().head(10))


funding_rounds.csv
Shape: (2818, 18)

Columns:
['round_id', 'startup_id', 'startup_name', 'funding_date', 'year', 'quarter', 'funding_stage', 'amount_million_usd', 'pre_money_valuation_million_usd', 'post_money_valuation_million_usd', 'equity_dilution_pct', 'lead_investor', 'investor_type', 'num_investors', 'sector', 'country', 'region', 'city']

Dtypes:
round_id                             object
startup_id                           object
startup_name                         object
funding_date                         object
year                                  int64
quarter                              object
funding_stage                        object
amount_million_usd                  float64
pre_money_valuation_million_usd     float64
post_money_valuation_million_usd    float64
equity_dilution_pct                 float64
lead_investor                        object
investor_type                        object
num_investors                         int64
sector                    

,round_id,startup_id,startup_name,funding_date,year,quarter,funding_stage,amount_million_usd,pre_money_valuation_million_usd,post_money_valuation_million_usd,equity_dilution_pct,lead_investor,investor_type,num_investors,sector,country,region,city
0,R00001,S00001,DataTech,2023-03-22,2023,Q1,Seed,4.20,13.98,18.18,13.8,Redpoint Ventures,Angel Investor,1,Gaming & Entertainment,Mexico,South America,Mexico City
1,R00002,S00002,VertexPlatform,2025-01-19,2025,Q1,Pre-Seed,0.55,2.34,2.89,12.7,General Catalyst,Angel Investor,2,Artificial Intelligence,Indonesia,Asia,Surabaya
2,R00003,S00002,VertexPlatform,2021-07-14,2021,Q3,Series A,15.35,176.14,191.49,18.4,Index Ventures,Corporate VC,3,Artificial Intelligence,Indonesia,Asia,Surabaya


Match rate: 100.0%

Unmatched lead_investor values (top 10):
Series([], Name: count, dtype: int64)


In [58]:
df_startups = inspect("startups.csv")

# Every funding round should point to a real startup_id
orphan_rounds = df_funding[~df_funding['startup_id'].isin(df_startups['startup_id'])]
print(f"Orphan funding rounds (no matching startup): {len(orphan_rounds)}")


startups.csv
Shape: (1747, 10)

Columns:
['startup_id', 'startup_name', 'sector', 'country', 'region', 'city', 'founded_year', 'employee_count', 'female_founder', 'revenue_status']

Dtypes:
startup_id        object
startup_name      object
sector            object
country           object
region            object
city              object
founded_year       int64
employee_count     int64
female_founder     int64
revenue_status    object
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,startup_id,startup_name,sector,country,region,city,founded_year,employee_count,female_founder,revenue_status
0,S00001,DataTech,Gaming & Entertainment,Mexico,South America,Mexico City,2020,9,0,Profitable
1,S00002,VertexPlatform,Artificial Intelligence,Indonesia,Asia,Surabaya,2024,2,1,Pre-Revenue
2,S00003,KiteAI,SaaS / Enterprise Software,Germany,Europe,Munich,2019,4,0,Pre-Revenue


Orphan funding rounds (no matching startup): 0


In [59]:
df_unicorns = inspect("unicorn_companies.csv")  # re-run if kernel lost it

overlap = set(df_unicorns['Company']).intersection(set(df_cb_clean['Company']))
print(f"unicorn_companies.csv: {len(df_unicorns)} companies")
print(f"CB Insights (cleaned): {len(df_cb_clean)} companies")
print(f"Overlapping company names: {len(overlap)}")


unicorn_companies.csv
Shape: (1500, 11)

Columns:
['Company', 'Valuation ($B)', 'Date Joined', 'Country', 'City', 'Industry', 'Select Investors', 'Founded Year', 'Total Raised ($B)', 'Financial Stage', 'Investors Count']

Dtypes:
Company               object
Valuation ($B)       float64
Date Joined           object
Country               object
City                  object
Industry              object
Select Investors      object
Founded Year           int64
Total Raised ($B)    float64
Financial Stage       object
Investors Count        int64
dtype: object

Null counts:
Series([], dtype: int64)

Sample rows:


,Company,Valuation ($B),Date Joined,Country,City,Industry,Select Investors,Founded Year,Total Raised ($B),Financial Stage,Investors Count
0,Grid Group94,1.58,2026-12-04,United States,Los Angeles,Data management & analytics,"DST Global, Kleiner Perkins Caufield & Byers, ...",2016,0.305,Series E,5
1,Meta Labs,2.55,2026-11-26,Germany,Munich,Fintech,Y Combinator,2022,0.528,Series D,12
2,Link Group,2.32,2026-10-14,Israel,Tel Aviv,Health,"New Enterprise Associates, DST Global, Alibaba...",2015,0.940,Series C,16


unicorn_companies.csv: 1500 companies
CB Insights (cleaned): 1360 companies
Overlapping company names: 1


In [ ]:
def standardize(df, name_col, industry_col, country_col, amount_col, stage_col, source_name):
    return pd.DataFrame({
        'startup_name': df[name_col],
        'industry': df[industry_col],
        'country': df[country_col],
        'funding_amount_usd': df[amount_col],
        'funding_stage': df[stage_col] if stage_col else None,
        'source_dataset': source_name
    })

# Example for one file - you'll write one call per Family D file
std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country', 
                     'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')

# Repeat for each Family D file, then:
supplementary = pd.concat([std_1, std_2, std_3, ...], ignore_index=True)

In [27]:
# Real unicorns should include well-known names. Check if they're actually there.
known_real_unicorns = ['OpenAI', 'ByteDance', 'SpaceX', 'Stripe', 'Anthropic', 'Canva', 'Databricks']
found = df_unicorns[df_unicorns['Company'].isin(known_real_unicorns)]
print(found[['Company', 'Valuation ($B)']])
print(f"\nMax Date Joined: {df_unicorns['Date Joined'].max()}")
print(f"Any future dates? {(pd.to_datetime(df_unicorns['Date Joined']) > pd.Timestamp.now()).sum()}")

Empty DataFrame
Columns: [Company, Valuation ($B)]
Index: []

Max Date Joined: 2026-12-04
Any future dates? 9


In [61]:
df_startup_funding = pd.read_csv(os.path.join(RAW_DIR, "startup_funding_dataset (1).csv"))
print(df_startup_funding.shape)
df_startup_funding.head()

(2000, 7)


,Startup Name,Industry,Country,Funding Stage,Amount Raised (USD),Funding Date,Number of Employees
0,"Williamson, Greer and Clark",SaaS,Australia,Seed,304706.0,2016-07-03,386
1,"Bradford, Green and Miranda",Finance,India,IPO,641212096.0,2025-08-10,474
2,Kelly LLC,AI,UK,Series A,4149256.0,2024-11-13,220
3,Riggs-Wells,Health,France,IPO,940259054.0,2018-10-03,14
4,"Walters, Edwards and Welch",E-commerce,France,Seed,419487.0,2021-12-27,137


In [62]:
df_indian = pd.read_csv(os.path.join(RAW_DIR, "Indian_Startup_Funding_Dataset.csv"))
df_recent_india = pd.read_csv(os.path.join(RAW_DIR, "Recently Funded Startups In India 2026.csv"))
# df_ai and df_2025 you already have from earlier cells - reuse them
# df_unicorns you already have too - now goes into Family D as well

In [65]:
def clean_dollar_string(val):
    if pd.isna(val):
        return None
    return float(str(val).replace('$', '').replace(',', ''))

df_recent_india['funding_amount_clean'] = df_recent_india['Funding Amount (USD)'].apply(clean_dollar_string)

In [73]:
df_2025 = pd.read_csv(os.path.join(RAW_DIR, "Startup_funding_2025.csv"))
df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)

df_recent_india = pd.read_csv(os.path.join(RAW_DIR, "Recently Funded Startups In India 2026.csv"))
df_recent_india['funding_amount_clean'] = df_recent_india['Funding Amount (USD)'].apply(clean_dollar_string)

std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country',
                     'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')
std_2 = standardize(df_ai, 'Startup_Name', 'Industry_AI_Application', 'Country',
                     'Funding_Amount ($M)', 'Funding_Stage', 'ai_startup_funding')
std_2['funding_amount_usd'] = std_2['funding_amount_usd'] * 1e6

std_3 = standardize(df_2025, 'Company', 'Sector', 'Headquarters',
                     'amount_usd', 'Funding_Round_Type', 'startup_funding_2025')

std_4 = standardize(df_indian, 'Startup_Name', 'Industry', 'State',
                     'Funding_Amount_USD', 'Funding_Stage', 'indian_startup_funding')

std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country',
                     'funding_amount_clean', 'Funding Type', 'recent_india_2026')

std_6 = standardize(df_unicorns, 'Company', 'Industry', 'Country',
                     'Total Raised ($B)', 'Financial Stage', 'unicorn_companies_synthetic')
std_6['funding_amount_usd'] = std_6['funding_amount_usd'] * 1e9

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)
print(supplementary.shape)
print(supplementary['source_dataset'].value_counts())
supplementary.head(10)

(4847, 6)
source_dataset
startup_funding_dataset        2000
unicorn_companies_synthetic    1500
startup_funding_2025            887
indian_startup_funding          300
recent_india_2026               100
ai_startup_funding               60
Name: count, dtype: int64


,startup_name,industry,country,funding_amount_usd,funding_stage,source_dataset
0,"Williamson, Greer and Clark",SaaS,Australia,304706.0,Seed,startup_funding_dataset
1,"Bradford, Green and Miranda",Finance,India,641212096.0,IPO,startup_funding_dataset
2,Kelly LLC,AI,UK,4149256.0,Series A,startup_funding_dataset
3,Riggs-Wells,Health,France,940259054.0,IPO,startup_funding_dataset
4,"Walters, Edwards and Welch",E-commerce,France,419487.0,Seed,startup_funding_dataset
5,"Martinez, Miller and Valdez",AI,UK,4356688.0,Series A,startup_funding_dataset
6,Garcia Inc,Education,USA,48615111.0,Series B,startup_funding_dataset
7,"Castro, Brown and Anderson",Finance,Netherlands,246306446.0,IPO,startup_funding_dataset
8,"Thompson, Orozco and Johnson",Gaming,Germany,387248.0,Seed,startup_funding_dataset
9,"Drake, Nelson and Smith",SaaS,USA,171659162.0,Series C,startup_funding_dataset


In [74]:
std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country',
                     'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')

std_2 = standardize(df_ai, 'Startup_Name', 'Industry_AI_Application', 'Country',
                     'Funding_Amount ($M)', 'Funding_Stage', 'ai_startup_funding')
std_2['funding_amount_usd'] = std_2['funding_amount_usd'] * 1e6  # this file is in $M, not raw USD

std_3 = standardize(df_2025, 'Company', 'Sector', 'Headquarters',
                     'amount_usd', 'Funding_Round_Type', 'startup_funding_2025')

std_4 = standardize(df_indian, 'Startup_Name', 'Industry', 'State',
                     'Funding_Amount_USD', 'Funding_Stage', 'indian_startup_funding')

std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country',
                     'funding_amount_clean', 'Funding Type', 'recent_india_2026')

std_6 = standardize(df_unicorns, 'Company', 'Industry', 'Country',
                     'Total Raised ($B)', 'Financial Stage', 'unicorn_companies_synthetic')
std_6['funding_amount_usd'] = std_6['funding_amount_usd'] * 1e9  # this file is in $B

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)
print(supplementary.shape)
print(supplementary['source_dataset'].value_counts())
supplementary.head(10)

(4847, 6)
source_dataset
startup_funding_dataset        2000
unicorn_companies_synthetic    1500
startup_funding_2025            887
indian_startup_funding          300
recent_india_2026               100
ai_startup_funding               60
Name: count, dtype: int64


,startup_name,industry,country,funding_amount_usd,funding_stage,source_dataset
0,"Williamson, Greer and Clark",SaaS,Australia,304706.0,Seed,startup_funding_dataset
1,"Bradford, Green and Miranda",Finance,India,641212096.0,IPO,startup_funding_dataset
2,Kelly LLC,AI,UK,4149256.0,Series A,startup_funding_dataset
3,Riggs-Wells,Health,France,940259054.0,IPO,startup_funding_dataset
4,"Walters, Edwards and Welch",E-commerce,France,419487.0,Seed,startup_funding_dataset
5,"Martinez, Miller and Valdez",AI,UK,4356688.0,Series A,startup_funding_dataset
6,Garcia Inc,Education,USA,48615111.0,Series B,startup_funding_dataset
7,"Castro, Brown and Anderson",Finance,Netherlands,246306446.0,IPO,startup_funding_dataset
8,"Thompson, Orozco and Johnson",Gaming,Germany,387248.0,Seed,startup_funding_dataset
9,"Drake, Nelson and Smith",SaaS,USA,171659162.0,Series C,startup_funding_dataset


In [76]:
supplementary.groupby('source_dataset')['funding_amount_usd'].describe()[['min', '50%', 'max']]

,min,50%,max
source_dataset,,,
ai_startup_funding,20000000.0,150000000.0,1.000000e+09
indian_startup_funding,100000.0,24700000.0,4.950000e+07
recent_india_2026,82941.0,3000000.0,3.045279e+08
startup_funding_2025,11300.0,4000000.0,1.000000e+09
startup_funding_dataset,50413.0,32465171.0,9.999670e+08
unicorn_companies_synthetic,111000000.0,705000000.0,6.065300e+10


In [77]:
scale_check = supplementary.groupby('source_dataset')['funding_amount_usd'].describe()[['min', '50%', 'max']]
scale_check = scale_check.rename(columns={'50%': 'median'})
scale_check

,min,median,max
source_dataset,,,
ai_startup_funding,20000000.0,150000000.0,1.000000e+09
indian_startup_funding,100000.0,24700000.0,4.950000e+07
recent_india_2026,82941.0,3000000.0,3.045279e+08
startup_funding_2025,11300.0,4000000.0,1.000000e+09
startup_funding_dataset,50413.0,32465171.0,9.999670e+08
unicorn_companies_synthetic,111000000.0,705000000.0,6.065300e+10


In [78]:
os.makedirs("../data/processed", exist_ok=True)

df_startups.to_csv("../data/processed/dim_startup.csv", index=False)
df_investors.to_csv("../data/processed/dim_investor.csv", index=False)
df_funding.to_csv("../data/processed/fact_funding_rounds.csv", index=False)
df_cb_clean.to_csv("../data/processed/ref_real_unicorns.csv", index=False)
supplementary.to_csv("../data/processed/supplementary_funding_events.csv", index=False)
# ml_training_set is just startup_success_dataset.csv as-is — copy it too
pd.read_csv(os.path.join(RAW_DIR, "startup_success_dataset.csv")).to_csv("../data/processed/ml_training_set.csv", index=False)

print("All processed tables saved.")

All processed tables saved.


In [79]:
# Check the actual dtype of the column in the recent_india_2026 slice
recent_slice = supplementary[supplementary['source_dataset'] == 'recent_india_2026']
print(recent_slice['funding_amount_usd'].dtype)
print(recent_slice['funding_amount_usd'].head(10))

float64
3247    1000000.0
3248    9803025.0
3249    4353085.0
3250          NaN
3251    5446125.0
3252          NaN
3253    3534664.0
3254          NaN
3255    2728575.0
3256    2396295.0
Name: funding_amount_usd, dtype: float64


In [80]:
# Find any non-numeric leftovers in the source column before standardize() ran
bad_rows = df_recent_india[~df_recent_india['funding_amount_clean'].apply(lambda x: isinstance(x, (float, int)) or pd.isna(x))]
print(bad_rows[['Name', 'Funding Amount (USD)', 'funding_amount_clean']])

Empty DataFrame
Columns: [Name, Funding Amount (USD), funding_amount_clean]
Index: []


In [81]:
df_recent_india['funding_amount_clean'] = pd.to_numeric(df_recent_india['funding_amount_clean'], errors='coerce')

In [82]:
std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country',
                     'funding_amount_clean', 'Funding Type', 'recent_india_2026')

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)
supplementary.groupby('source_dataset')['funding_amount_usd'].describe()[['min', '50%', 'max']]

,min,50%,max
source_dataset,,,
ai_startup_funding,20000000.0,150000000.0,1.000000e+09
indian_startup_funding,100000.0,24700000.0,4.950000e+07
recent_india_2026,82941.0,3000000.0,3.045279e+08
startup_funding_2025,11300.0,4000000.0,1.000000e+09
startup_funding_dataset,50413.0,32465171.0,9.999670e+08
unicorn_companies_synthetic,111000000.0,705000000.0,6.065300e+10


In [ ]:
# ===== FULL RELOAD — run after any kernel restart =====

# Core relational tables
df_startups = pd.read_csv(os.path.join(RAW_DIR, "startups.csv"))

df_investors = pd.read_csv(os.path.join(RAW_DIR, "investors.csv"))
df_investors = df_investors.reset_index(drop=True)
df_investors.insert(0, 'investor_id', ['INV' + str(i+1).zfill(4) for i in range(len(df_investors))])

df_funding = pd.read_csv(os.path.join(RAW_DIR, "funding_rounds.csv"))
investor_lookup = dict(zip(df_investors['investor_name'], df_investors['investor_id']))
df_funding['investor_id'] = df_funding['lead_investor'].map(investor_lookup)

# CB Insights real unicorns (cleaned)
df_cb = pd.read_excel(os.path.join(RAW_DIR, "CB-Insights_Global-Unicorn-Club.xlsx"), skiprows=2)
df_cb = df_cb.drop(columns=['Unnamed: 0'])
df_cb_clean = df_cb[df_cb['Valuation ($B)'].notna()].reset_index(drop=True)

# Family D sources
df_startup_funding = pd.read_csv(os.path.join(RAW_DIR, "startup_funding_dataset (1).csv"))

with open(os.path.join(RAW_DIR, "ai_startup_funding.csv"), 'r', encoding='utf-8') as f:
    lines = f.readlines()
cleaned_lines = []
for line in lines:
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    line = line.replace('""', '"')
    cleaned_lines.append(line)
df_ai = pd.read_csv(StringIO("\n".join(cleaned_lines)))

df_2025 = pd.read_csv(os.path.join(RAW_DIR, "Startup_funding_2025.csv"))
df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)

df_indian = pd.read_csv(os.path.join(RAW_DIR, "Indian_Startup_Funding_Dataset.csv"))

df_recent_india = pd.read_csv(os.path.join(RAW_DIR, "Recently Funded Startups In India 2026.csv"))
df_recent_india['funding_amount_clean'] = df_recent_india['Funding Amount (USD)'].apply(clean_dollar_string)
df_recent_india['funding_amount_clean'] = pd.to_numeric(df_recent_india['funding_amount_clean'], errors='coerce')

df_unicorns = pd.read_csv(os.path.join(RAW_DIR, "unicorn_companies.csv"))

# Rebuild supplementary
std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country', 'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')
std_2 = standardize(df_ai, 'Startup_Name', 'Industry_AI_Application', 'Country', 'Funding_Amount ($M)', 'Funding_Stage', 'ai_startup_funding')
std_2['funding_amount_usd'] = std_2['funding_amount_usd'] * 1e6
std_3 = standardize(df_2025, 'Company', 'Sector', 'Headquarters', 'amount_usd', 'Funding_Round_Type', 'startup_funding_2025')
std_4 = standardize(df_indian, 'Startup_Name', 'Industry', 'State', 'Funding_Amount_USD', 'Funding_Stage', 'indian_startup_funding')
std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country', 'funding_amount_clean', 'Funding Type', 'recent_india_2026')
std_6 = standardize(df_unicorns, 'Company', 'Industry', 'Country', 'Total Raised ($B)', 'Financial Stage', 'unicorn_companies_synthetic')
std_6['funding_amount_usd'] = std_6['funding_amount_usd'] * 1e9

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)

print("All dataframes reloaded:")
for name in ['df_startups', 'df_investors', 'df_funding', 'df_cb_clean', 'supplementary']:
    print(f"  {name}: {eval(name).shape}")
# ========================================================

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

df_startups.to_csv("../data/processed/dim_startup.csv", index=False)
df_investors.to_csv("../data/processed/dim_investor.csv", index=False)
df_funding.to_csv("../data/processed/fact_funding_rounds.csv", index=False)
df_cb_clean.to_csv("../data/processed/ref_real_unicorns.csv", index=False)
supplementary.to_csv("../data/processed/supplementary_funding_events.csv", index=False)
pd.read_csv(os.path.join(RAW_DIR, "startup_success_dataset.csv")).to_csv("../data/processed/ml_training_set.csv", index=False)

print("All processed tables saved.")

In [11]:
import os
print("Current working directory:", os.getcwd())

Current working directory: /Users/khushithakur/Analytics/Startup Investment Intelligence Project


In [12]:
print("Contents of cwd:")
print(os.listdir('.'))

print("\nContents of parent folder:")
print(os.listdir('..'))

Contents of cwd:
['01_data_exploration.ipynb', '00_Environment_Setup.ipynb', '.ipynb_checkpoints', 'data']

Contents of parent folder:
['Customer Journey Funnel Analytics project', '.DS_Store', 'Startup investment intelligence', '.gitignore', '.virtual_documents', 'Startup Investment Intelligence Project', '.ipynb_checkpoints', '.git', 'data']


In [15]:
RAW_DIR = "data/raw"
print(os.listdir(RAW_DIR))

['CB-Insights_Global-Unicorn-Club.xlsx', 'funding_rounds.csv', 'investors.csv', 'startup_data.csv', 'unicorn_companies.csv', 'Indian_Startup_Funding_Dataset.csv', 'Startup_funding_2025.csv', 'quarterly_summary.csv', 'yc_startups.csv', 'startup_success_dataset.csv', 'openvc_investors.csv', '.ipynb_checkpoints', 'Recently Funded Startups In India 2026.csv', 'startups.csv', 'startup_funding_dataset (1).csv', 'ai_startup_funding.csv']


In [16]:
print(os.listdir(".."))
# Check if the other folder also has data in it
print(os.listdir("../Startup investment intelligence"))

['Customer Journey Funnel Analytics project', '.DS_Store', 'Startup investment intelligence', '.gitignore', '.virtual_documents', 'Startup Investment Intelligence Project', '.ipynb_checkpoints', '.git', 'data']
['.DS_Store', 'dataset_inventory.csv', '.ipynb_checkpoints', 'scrapers', '00_dataset_inventory.ipynb', 'data']


In [17]:
RAW_DIR = "data/raw"

# then run the full reload cell from before, unchanged

In [19]:
RAW_DIR = "data/raw"

# ===== FULL RELOAD =====
df_startups = pd.read_csv(os.path.join(RAW_DIR, "startups.csv"))

df_investors = pd.read_csv(os.path.join(RAW_DIR, "investors.csv"))
df_investors = df_investors.reset_index(drop=True)
df_investors.insert(0, 'investor_id', ['INV' + str(i+1).zfill(4) for i in range(len(df_investors))])

df_funding = pd.read_csv(os.path.join(RAW_DIR, "funding_rounds.csv"))
investor_lookup = dict(zip(df_investors['investor_name'], df_investors['investor_id']))
df_funding['investor_id'] = df_funding['lead_investor'].map(investor_lookup)

df_cb = pd.read_excel(os.path.join(RAW_DIR, "CB-Insights_Global-Unicorn-Club.xlsx"), skiprows=2)
df_cb = df_cb.drop(columns=['Unnamed: 0'])
df_cb_clean = df_cb[df_cb['Valuation ($B)'].notna()].reset_index(drop=True)

df_startup_funding = pd.read_csv(os.path.join(RAW_DIR, "startup_funding_dataset (1).csv"))

with open(os.path.join(RAW_DIR, "ai_startup_funding.csv"), 'r', encoding='utf-8') as f:
    lines = f.readlines()
cleaned_lines = []
for line in lines:
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    line = line.replace('""', '"')
    cleaned_lines.append(line)
df_ai = pd.read_csv(StringIO("\n".join(cleaned_lines)))

df_2025 = pd.read_csv(os.path.join(RAW_DIR, "Startup_funding_2025.csv"))
df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)

df_indian = pd.read_csv(os.path.join(RAW_DIR, "Indian_Startup_Funding_Dataset.csv"))

df_recent_india = pd.read_csv(os.path.join(RAW_DIR, "Recently Funded Startups In India 2026.csv"))
df_recent_india['funding_amount_clean'] = df_recent_india['Funding Amount (USD)'].apply(clean_dollar_string)
df_recent_india['funding_amount_clean'] = pd.to_numeric(df_recent_india['funding_amount_clean'], errors='coerce')

df_unicorns = pd.read_csv(os.path.join(RAW_DIR, "unicorn_companies.csv"))

std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country', 'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')
std_2 = standardize(df_ai, 'Startup_Name', 'Industry_AI_Application', 'Country', 'Funding_Amount ($M)', 'Funding_Stage', 'ai_startup_funding')
std_2['funding_amount_usd'] = std_2['funding_amount_usd'] * 1e6
std_3 = standardize(df_2025, 'Company', 'Sector', 'Headquarters', 'amount_usd', 'Funding_Round_Type', 'startup_funding_2025')
std_4 = standardize(df_indian, 'Startup_Name', 'Industry', 'State', 'Funding_Amount_USD', 'Funding_Stage', 'indian_startup_funding')
std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country', 'funding_amount_clean', 'Funding Type', 'recent_india_2026')
std_6 = standardize(df_unicorns, 'Company', 'Industry', 'Country', 'Total Raised ($B)', 'Financial Stage', 'unicorn_companies_synthetic')
std_6['funding_amount_usd'] = std_6['funding_amount_usd'] * 1e9

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)

print("All dataframes reloaded:")
for name in ['df_startups', 'df_investors', 'df_funding', 'df_cb_clean', 'supplementary']:
    print(f"  {name}: {eval(name).shape}")

All dataframes reloaded:
  df_startups: (1747, 10)
  df_investors: (73, 11)
  df_funding: (2818, 19)
  df_cb_clean: (1360, 7)
  supplementary: (4847, 6)


In [20]:
os.makedirs("data/processed", exist_ok=True)

df_startups.to_csv("data/processed/dim_startup.csv", index=False)
df_investors.to_csv("data/processed/dim_investor.csv", index=False)
df_funding.to_csv("data/processed/fact_funding_rounds.csv", index=False)
df_cb_clean.to_csv("data/processed/ref_real_unicorns.csv", index=False)
supplementary.to_csv("data/processed/supplementary_funding_events.csv", index=False)
pd.read_csv(os.path.join(RAW_DIR, "startup_success_dataset.csv")).to_csv("data/processed/ml_training_set.csv", index=False)

print("All processed tables saved.")
print(os.listdir("data/processed"))

All processed tables saved.
['ref_real_unicorns.csv', 'ml_training_set.csv', 'fact_funding_rounds.csv', 'dim_startup.csv', 'supplementary_funding_events.csv', 'dim_investor.csv']


In [21]:
# ===== CELL 1: SETUP — run this first, every session =====
import pandas as pd
import numpy as np
import os
import re
from io import StringIO

RAW_DIR = "data/raw"


def inspect(filename, nrows=None):
    path = os.path.join(RAW_DIR, filename)
    if filename.endswith('.xlsx'):
        df = pd.read_excel(path, nrows=nrows)
    else:
        df = pd.read_csv(path, nrows=nrows, low_memory=False)

    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{list(df.columns)}")
    print(f"\nDtypes:\n{df.dtypes}")
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"\nSample rows:")
    display(df.head(3))
    return df


def parse_amount_to_usd(val, inr_to_usd=0.012):
    if pd.isna(val) or val == '-' or 'undisclosed' in str(val).lower():
        return None
    val = str(val).strip()
    if 'crore' in val.lower():
        num = float(re.search(r'[\d.]+', val).group())
        return num * 1e7 * inr_to_usd
    elif '$' in val:
        num = float(re.search(r'[\d.]+', val.replace(',', '')).group())
        if 'K' in val.upper():
            multiplier = 1e3
        elif 'M' in val.upper():
            multiplier = 1e6
        elif 'B' in val.upper():
            multiplier = 1e9
        else:
            multiplier = 1
        return num * multiplier
    return None


def clean_dollar_string(val):
    if pd.isna(val):
        return None
    return float(str(val).replace('$', '').replace(',', ''))


def standardize(df, name_col, industry_col, country_col, amount_col, stage_col, source_name):
    return pd.DataFrame({
        'startup_name': df[name_col],
        'industry': df[industry_col],
        'country': df[country_col],
        'funding_amount_usd': df[amount_col],
        'funding_stage': df[stage_col] if stage_col else None,
        'source_dataset': source_name
    })

print("Setup complete.")

Setup complete.


In [22]:
# ===== CELL 2: FULL RELOAD — run every session, right after Cell 1 =====

df_startups = pd.read_csv(os.path.join(RAW_DIR, "startups.csv"))

df_investors = pd.read_csv(os.path.join(RAW_DIR, "investors.csv"))
df_investors = df_investors.reset_index(drop=True)
df_investors.insert(0, 'investor_id', ['INV' + str(i+1).zfill(4) for i in range(len(df_investors))])

df_funding = pd.read_csv(os.path.join(RAW_DIR, "funding_rounds.csv"))
investor_lookup = dict(zip(df_investors['investor_name'], df_investors['investor_id']))
df_funding['investor_id'] = df_funding['lead_investor'].map(investor_lookup)

df_cb = pd.read_excel(os.path.join(RAW_DIR, "CB-Insights_Global-Unicorn-Club.xlsx"), skiprows=2)
df_cb = df_cb.drop(columns=['Unnamed: 0'])
df_cb_clean = df_cb[df_cb['Valuation ($B)'].notna()].reset_index(drop=True)

df_startup_funding = pd.read_csv(os.path.join(RAW_DIR, "startup_funding_dataset (1).csv"))

with open(os.path.join(RAW_DIR, "ai_startup_funding.csv"), 'r', encoding='utf-8') as f:
    lines = f.readlines()
cleaned_lines = []
for line in lines:
    line = line.strip()
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1]
    line = line.replace('""', '"')
    cleaned_lines.append(line)
df_ai = pd.read_csv(StringIO("\n".join(cleaned_lines)))

df_2025 = pd.read_csv(os.path.join(RAW_DIR, "Startup_funding_2025.csv"))
df_2025['amount_usd'] = df_2025['Amount'].apply(parse_amount_to_usd)

df_indian = pd.read_csv(os.path.join(RAW_DIR, "Indian_Startup_Funding_Dataset.csv"))

df_recent_india = pd.read_csv(os.path.join(RAW_DIR, "Recently Funded Startups In India 2026.csv"))
df_recent_india['funding_amount_clean'] = df_recent_india['Funding Amount (USD)'].apply(clean_dollar_string)
df_recent_india['funding_amount_clean'] = pd.to_numeric(df_recent_india['funding_amount_clean'], errors='coerce')

df_unicorns = pd.read_csv(os.path.join(RAW_DIR, "unicorn_companies.csv"))

std_1 = standardize(df_startup_funding, 'Startup Name', 'Industry', 'Country', 'Amount Raised (USD)', 'Funding Stage', 'startup_funding_dataset')
std_2 = standardize(df_ai, 'Startup_Name', 'Industry_AI_Application', 'Country', 'Funding_Amount ($M)', 'Funding_Stage', 'ai_startup_funding')
std_2['funding_amount_usd'] = std_2['funding_amount_usd'] * 1e6
std_3 = standardize(df_2025, 'Company', 'Sector', 'Headquarters', 'amount_usd', 'Funding_Round_Type', 'startup_funding_2025')
std_4 = standardize(df_indian, 'Startup_Name', 'Industry', 'State', 'Funding_Amount_USD', 'Funding_Stage', 'indian_startup_funding')
std_5 = standardize(df_recent_india, 'Name', 'Industry', 'Country', 'funding_amount_clean', 'Funding Type', 'recent_india_2026')
std_6 = standardize(df_unicorns, 'Company', 'Industry', 'Country', 'Total Raised ($B)', 'Financial Stage', 'unicorn_companies_synthetic')
std_6['funding_amount_usd'] = std_6['funding_amount_usd'] * 1e9

supplementary = pd.concat([std_1, std_2, std_3, std_4, std_5, std_6], ignore_index=True)

print("All dataframes reloaded:")
for name in ['df_startups', 'df_investors', 'df_funding', 'df_cb_clean', 'supplementary']:
    print(f"  {name}: {eval(name).shape}")

All dataframes reloaded:
  df_startups: (1747, 10)
  df_investors: (73, 11)
  df_funding: (2818, 19)
  df_cb_clean: (1360, 7)
  supplementary: (4847, 6)
